# License
This notebook is licensed under the MIT License. See the LICENSE file in the repository root for details.

In [ ]:
# Setup for Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q pandas
    print('Packages installed.')


# SQL for Data Analysts In-Depth

Structured Query Language (SQL) is the standard language for managing and manipulating relational databases. For a Data Analyst, it is often considered the most critical technical skill alongside Python/R.

In this notebook, we will:
1.  Create a temporary SQLite database in memory.
2.  Populate it with synthetic social science data (Participants and Survey Responses).
3.  Perform essential SQL operations: `SELECT`, `WHERE`, `JOIN`, `GROUP BY`.
4.  Explore advanced concepts: **Window Functions** and **CTEs**.
5.  Integrate SQL with Python using `pandas`.


In [ ]:
import sqlite3
import pandas as pd

# Create an in-memory SQLite database connection
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Create 'participants' table
cursor.execute('''
CREATE TABLE participants (
    participant_id INTEGER PRIMARY KEY,
    age INTEGER,
    gender TEXT,
    location TEXT
)
''')

# Create 'responses' table
cursor.execute('''
CREATE TABLE responses (
    response_id INTEGER PRIMARY KEY,
    participant_id INTEGER,
    survey_score INTEGER,
    completion_time_min INTEGER,
    FOREIGN KEY(participant_id) REFERENCES participants(participant_id)
)
''')

print("Tables created successfully.")

In [ ]:
# Insert dummy data
participants_data = [
    (1, 25, 'Female', 'New York'),
    (2, 30, 'Male', 'London'),
    (3, 22, 'Non-binary', 'Berlin'),
    (4, 45, 'Female', 'New York'),
    (5, 35, 'Male', 'Paris')
]

responses_data = [
    (1, 1, 85, 12),
    (2, 2, 90, 15),
    (3, 3, 78, 20),
    (4, 1, 88, 10),  # Participant 1 took it again (longitudinal?)
    (5, 4, 92, 14)
    # Participant 5 has no response yet
]

cursor.executemany('INSERT INTO participants VALUES (?, ?, ?, ?)', participants_data)
cursor.executemany('INSERT INTO responses VALUES (?, ?, ?, ?)', responses_data)
conn.commit()

print("Data inserted successfully.")

## 1. Basic SELECT Queries

The `SELECT` statement is used to fetch data from a database.

**Syntax:** `SELECT column1, column2 FROM table_name;`

In [ ]:
# Select everything from participants
query = "SELECT * FROM participants"
df = pd.read_sql_query(query, conn)
print("--- All Participants ---")
print(df)

# Select specific columns with a condition (WHERE)
query = "SELECT participant_id, age FROM participants WHERE age > 25"
df = pd.read_sql_query(query, conn)
print("\n--- Participants Older than 25 ---")
print(df)

## 2. JOIN Operations

To analyze data from multiple tables, we use `JOIN`. 

*   `INNER JOIN`: Returns records that have matching values in both tables.
*   `LEFT JOIN`: Returns all records from the left table, and the matched records from the right table.


In [ ]:
# Inner Join: Only show participants who have a response
query = '''
SELECT 
    p.participant_id, 
    p.gender, 
    r.survey_score
FROM participants p
INNER JOIN responses r ON p.participant_id = r.participant_id
'''
df = pd.read_sql_query(query, conn)
print("--- Inner Join (Participants with Responses) ---")
print(df)

# Left Join: Show all participants, even if they have no response (NaN will appear)
query = '''
SELECT 
    p.participant_id, 
    p.location, 
    r.survey_score
FROM participants p
LEFT JOIN responses r ON p.participant_id = r.participant_id
'''
df = pd.read_sql_query(query, conn)
print("\n--- Left Join (All Participants) ---")
print(df)

## 3. Aggregation (GROUP BY)

We can calculate summary statistics (Mean, Count, Max) using `GROUP BY`.

In [ ]:
# Average survey score by City (Location)
# Note: We need to join first to link location (participants) with score (responses)
query = '''
SELECT 
    p.location,
    COUNT(r.response_id) as response_count,
    AVG(r.survey_score) as avg_score
FROM participants p
JOIN responses r ON p.participant_id = r.participant_id
GROUP BY p.location
'''
df = pd.read_sql_query(query, conn)
print("--- Aggregation by Location ---")
print(df)

## 4. Window Functions (Advanced)

Window functions perform calculations across a set of table rows that are somehow related to the current row. Unlike `GROUP BY`, they do not collapse rows.

*   `RANK()`: Assigns a rank to each row within a partition.
*   `AVG() OVER()`: Calculates a moving average or cumulative average.

In [ ]:
# Rank participants by survey_score within their location
query = '''
SELECT 
    p.location,
    p.participant_id,
    r.survey_score,
    RANK() OVER (PARTITION BY p.location ORDER BY r.survey_score DESC) as rank_in_city
FROM participants p
JOIN responses r ON p.participant_id = r.participant_id
'''
df = pd.read_sql_query(query, conn)
print("--- Ranking within Location (Window Function) ---")
print(df)

## 5. Common Table Expressions (CTEs)

CTEs allow you to define a temporary result set that you can reference within a `SELECT`, `INSERT`, `UPDATE`, or `DELETE` statement. They make complex queries more readable.

**Syntax:** `WITH cte_name AS (SELECT ...) SELECT * FROM cte_name;`

In [ ]:
# Use a CTE to first calculate average score per city, then filter for high-performing cities
query = '''
WITH CityStats AS (
    SELECT 
        p.location,
        AVG(r.survey_score) as avg_score
    FROM participants p
    JOIN responses r ON p.participant_id = r.participant_id
    GROUP BY p.location
)
SELECT * 
FROM CityStats
WHERE avg_score > 80
ORDER BY avg_score DESC
'''
df = pd.read_sql_query(query, conn)
print("--- High Performing Cities (Using CTE) ---")
print(df)

## 6. Closing Connection
Always close the database connection when finished.

In [ ]:
conn.close()